# 03 — Rolling Walk-Forward Backtest

Compares out-of-sample performance of the Global Minimum Variance portfolio under three
covariance estimators:
- **Sample**: plain sample covariance
- **Ledoit-Wolf**: analytical shrinkage toward scaled identity
- **James-Stein**: shrinkage toward single-factor (market) structure

Also includes **1/N equal-weight** as a benchmark.

Statistical significance of performance differences is assessed via the Diebold-Mariano test.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.estimation.sample      import sample_covariance
from src.estimation.ledoit_wolf import ledoit_wolf
from src.estimation.james_stein import james_stein
from src.backtest.rolling        import run_backtest
from src.backtest.metrics        import summary_table, diebold_mariano_test

In [ ]:
excess = pd.read_csv("../data/excess_returns_weekly.csv", index_col=0, parse_dates=True)
print(f"Loaded: {excess.shape[0]} weeks × {excess.shape[1]} stocks")

## 1. Run backtest

In [ ]:
# Wrap james_stein to return only the covariance (drop the alpha scalar)
def js_estimator(returns):
    cov, _ = james_stein(returns)
    return cov

estimators = {
    "Sample": sample_covariance,
    "Ledoit-Wolf": ledoit_wolf,
    "James-Stein": js_estimator,
}

WINDOW = 52  # weeks of history used per estimation

portfolio_returns = run_backtest(excess, estimators, window=WINDOW)

# Add 1/N benchmark
N = excess.shape[1]
portfolio_returns["1/N"] = excess.iloc[WINDOW + 1:].mean(axis=1).values[:len(portfolio_returns)]

print(f"Backtest period: {portfolio_returns.index[0].date()} → {portfolio_returns.index[-1].date()}")
print(f"Total weeks: {len(portfolio_returns)}")

## 2. Performance summary

In [ ]:
summary_table(portfolio_returns)

## 3. Cumulative return curves

In [ ]:
cum_returns = (1 + portfolio_returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
for col in cum_returns.columns:
    style = '--' if col == '1/N' else '-'
    ax.plot(cum_returns.index, cum_returns[col], label=col, linestyle=style)

ax.set_title("Cumulative Out-of-Sample Returns — GMV Portfolio (Rolling 52-week window)")
ax.set_ylabel("Cumulative return (gross)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Realized variance over time

In [ ]:
rolling_var = portfolio_returns.rolling(26).var() * 52  # annualized

fig, ax = plt.subplots(figsize=(12, 4))
for col in rolling_var.columns:
    style = '--' if col == '1/N' else '-'
    ax.plot(rolling_var.index, rolling_var[col], label=col, linestyle=style)

ax.set_title("Rolling 26-week Realized Annualized Variance")
ax.set_ylabel("Annualized variance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Diebold-Mariano significance tests

Tests whether the difference in realized portfolio variance between strategies is statistically significant.
H0: equal out-of-sample loss. Negative DM statistic → first strategy has lower realized variance.

In [ ]:
# Compare each shrinkage strategy against Sample as baseline
baseline = portfolio_returns["Sample"]

for name in ["Ledoit-Wolf", "James-Stein", "1/N"]:
    result = diebold_mariano_test(baseline, portfolio_returns[name])
    print(f"Sample vs {name}:")
    print(f"  DM statistic = {result['statistic']},  p-value = {result['p_value']}")
    print(f"  {result['interpretation']}")
    print()